# WSP bản nhanh: 10 mẫu, nhãn theo điểm, 3 vòng sửa (khoảng 1 giờ)

Model: `Ma7ee7/Qwen3.8_4B_Distilled` (revision `f33da883e0dadd9a4026556274e87489cd544d77`). Cài đặt: **GPU T4 x2**, **Internet On**. Chạy **Run All** (tương tác).
Input: dataset riêng tư chứa `zaic_code_20261111_v5.zip` và `zaic_data_noaudio_20261107.zip`.
Đường dẫn cần kiểm: ô "Thiết lập đường dẫn" (dòng CODE_ZIP). Các đường dẫn khác tự tìm dưới `/kaggle/input`.

Vì sao có bản này: lần chạy 09/11 không ra bản sửa vì practice có `bad 0` (model không bị cắt/lặp thinking),
không phải vì hết giờ. Bản này đổi luật nhãn (`--label-rule score`): **bad = mẫu điểm 0, good = mẫu điểm 1,0**.
Tập chạy: 10 mẫu practice (`fast10_20261111.txt`, 5 mẫu mỗi GPU), chọn từ 20 mẫu QUICK ngày 09/11 gồm 2 mẫu đúng và 8 mẫu sai.

1. Vòng 0 (model gốc) trên 10 mẫu, gán nhãn, tìm hướng (`dirs.pt`).
2. `auto_search.py`: 3 vòng sửa, mỗi vòng trừ hướng khỏi 64 cột MLP ở 1 tầng (3 tầng có Cohen's d cao nhất).
3. Bảng: từng bản sửa, điểm so với vòng 0, GIỮ hay bỏ theo luật cố định (tăng >= 0,01, gen_tokens <= 105%).
4. Nếu có bản được giữ và còn giờ: kiểm trên 10 mẫu dev.

Giới hạn: 10 mẫu là quá ít để kết luận go/no-go; 10 mẫu này được chọn theo kết quả cũ nên điểm vòng 0 không đại diện.
Kết quả: `/kaggle/working/wsp_results.zip`. Không gọi API ngoài; không chạy trên test BTC.

In [ ]:
# ===== CẤU HÌNH =====
QUICK = True          # bản nhanh luôn dùng tập nhỏ
MODE = "on"
BUDGET = 3            # số vòng sửa (mỗi vòng ước 8-10 phút, CHƯA XÁC MINH)
TOP_LAYERS = 3        # số tầng ứng viên (pha A thử từng tầng)
HOURS = 1.3           # không bắt đầu vòng sửa mới sau mốc này (giờ, tính từ lúc chạy ô 2)
DEV_CHECK_BEFORE_H = 1.0  # chỉ kiểm dev (2 lượt x 10 mẫu) nếu lúc đó chưa quá mốc này

In [ ]:
# ĐƯỜNG DẪN INPUT (chỉ sửa nếu cần):
#  - CODE_ZIP: phải TRÙNG tên file zip mã đã upload vào dataset (hiện: zaic_code_20261111_v5.zip).
#  - Dữ liệu (zaic_data_noaudio_20261107.zip) và train/train.jsonl: không cần sửa, ô tự tìm dưới /kaggle/input/**.
#  - Nếu assert báo lỗi: kiểm tra dataset có đúng tên zip ở trên và là bản v5 (có --label-rule).
import glob, os, shutil, subprocess, sys, zipfile, json, time
NB_T0 = time.time()  # mốc bắt đầu notebook, để tính hạn giờ
W = "/kaggle/working"; T = "/tmp"
CODE_ZIP = "zaic_code_20261111_v5.zip"
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):   # trường hợp Kaggle không tự giải nén
    n = os.path.basename(z)
    if "zaic_code" in n:
        zipfile.ZipFile(z).extractall(f"{T}/code_src/{n[:-4]}")
    elif "zaic_data" in n:
        zipfile.ZipFile(z).extractall(f"{T}/data_root")
cands = sorted(glob.glob(f"{T}/code_src/**/optimize/wsp/auto_search.py", recursive=True)
               + glob.glob("/kaggle/input/**/optimize/wsp/auto_search.py", recursive=True))
cands = [p for p in cands if CODE_ZIP[:-4] in p] or cands   # ưu tiên đúng bản zip, tránh lấy nhầm bản cũ
assert cands, f"không thấy mã nguồn (optimize/wsp/auto_search.py): kiểm tra dataset có {CODE_ZIP}"
assert "--deadline-ts" in open(cands[0], encoding="utf-8").read(), f"mã nguồn cũ ({cands[0]}): cần {CODE_ZIP}"
REPO = W + "/ZAIC_2026"
shutil.copytree(os.path.dirname(os.path.dirname(os.path.dirname(cands[0]))), REPO, dirs_exist_ok=True)
tj = glob.glob(f"{T}/data_root/**/train/train.jsonl", recursive=True) or glob.glob("/kaggle/input/**/train/train.jsonl", recursive=True)
assert tj, "không thấy dữ liệu (train/train.jsonl)"
ROOT = os.path.dirname(os.path.dirname(tj[0]))
print("REPO =", REPO, "(từ", cands[0], ")"); print("ROOT =", ROOT, os.listdir(ROOT))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm==0.33.0   # bản đã chạy được trên Kaggle T4 ngày 09/11
import torch, importlib.metadata as md
DTYPE = "auto" if torch.cuda.get_device_capability(0) >= (8, 0) else "half"   # T4 không có bf16
GPUS = ",".join(str(i) for i in range(torch.cuda.device_count()))
print("torch", torch.__version__, "| vllm", md.version("vllm"), "| dtype:", DTYPE, "| GPU:", GPUS)

In [ ]:
from huggingface_hub import snapshot_download
# cùng bản với máy cá nhân (docs/sources.md, DECISIONS 09/11); để ở /tmp cho khỏi bị lưu vào Output
MODEL_ID, MODEL_REV = "Ma7ee7/Qwen3.8_4B_Distilled", "f33da883e0dadd9a4026556274e87489cd544d77"
M = snapshot_download(MODEL_ID, revision=MODEL_REV, local_dir=T + "/model")
print(M, sorted(os.listdir(M)))

In [ ]:
def run(args):
    cmd = [sys.executable] + args
    print(">>", " ".join(cmd), flush=True)
    t0 = time.time()
    subprocess.run(cmd, cwd=REPO, check=True)
    print(f"   xong sau {time.time() - t0:.0f}s | đã chạy {(time.time() - NB_T0) / 3600:.2f} giờ", flush=True)

def head(src, n, dst):
    ids = [l.strip() for l in open(f"{REPO}/{src}", encoding="utf-8") if l.strip()][:n]
    open(dst, "w", encoding="utf-8").write("\n".join(ids) + "\n"); return dst

def summary(p):
    s = json.load(open(p, encoding="utf-8")); s.pop("per_sample", None); return s

def pack(name="wsp_results.zip"):  # gọi sau mỗi bước: Commit bị dừng giữa chừng vẫn còn kết quả các bước đã xong
    out = f"{W}/{name}"
    files = glob.glob(f"{W}/dev_*/*") + glob.glob(f"{W}/search/*") + \
            [f"{W}/{x}" for x in ("dirs.pt", "dirs.stats.json", "best_spec.json", "run_config.json")]
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files:
            if os.path.isfile(f):
                z.write(f, os.path.relpath(f, W))
    print("->", out, round(os.path.getsize(out) / 1e6, 2), "MB", flush=True)

DEV = head("eval/splits/dev_ids.txt", 10, W + "/dev_quick.txt") if QUICK else f"{REPO}/eval/splits/dev_ids.txt"
PRACTICE = head("optimize/wsp/practice100_20261107.txt", 20, W + "/practice_quick.txt") if QUICK else f"{REPO}/optimize/wsp/practice100_20261107.txt"
K = ["--base-model", M, "--root", ROOT, "--llm-dtype", DTYPE, "--gpu-mem", "0.9", "--gpus", GPUS, "--model-out", T + "/wsp_model"]
S = f"{W}/search"
print("QUICK:", QUICK, "| MODE:", MODE, "| DEV:", DEV, "| PRACTICE:", PRACTICE)

In [ ]:
# Bước 1: vòng 0 trên 10 mẫu, nhãn theo điểm, tìm hướng
assert "--label-rule" in open(f"{REPO}/optimize/wsp/analyze.py", encoding="utf-8").read(), f"mã nguồn cũ: cần {CODE_ZIP}"
SEL = f"{REPO}/optimize/wsp/fast10_20261111.txt"
json.dump({"notebook": "fast", "mode": MODE, "model": MODEL_ID, "revision": MODEL_REV, "code_zip": CODE_ZIP,
           "label_rule": "score", "select": os.path.basename(SEL), "budget": BUDGET, "top_layers": TOP_LAYERS},
          open(W + "/run_config.json", "w"), indent=1)
run(["optimize/wsp/run_round.py", "--round", "0", "--spec", "none", "--think", MODE, "--ids", SEL, "--work", S,
     "--label-rule", "score"] + K)
exp = S + "/r00.export.jsonl"
recs = [json.loads(l) for l in open(exp, encoding="utf-8")]
NB, NG = sum(r["label"] == "bad" for r in recs), sum(r["label"] == "good" for r in recs)
print(f"nhãn: bad {NB} | good {NG} |", [(r["id"], r["label"], r["score"]) for r in recs])
HAVE_DIRS = NB >= 2 and NG >= 2
if HAVE_DIRS:
    run(["optimize/wsp/directions.py", "--model", M, "--root", ROOT, "--export", exp, "--think", MODE,
         "--out", W + "/dirs.pt", "--stats", W + "/dirs.stats.json"])
    print(json.dumps(json.load(open(W + "/dirs.stats.json"))["layers"][:6], indent=0))
else:
    print(f"KHÔNG ĐỦ NHÃN (bad {NB}, good {NG}): không tìm được hướng. Gửi lại wsp_results.zip.")
pack()

In [ ]:
# Bước 2: 3 vòng sửa (pha A: từng tầng một)
DEADLINE = NB_T0 + HOURS * 3600
if HAVE_DIRS:
    run(["optimize/wsp/auto_search.py", "--base-model", M, "--root", ROOT, "--dirs", W + "/dirs.pt", "--stats", W + "/dirs.stats.json",
         "--work", S, "--model-out", T + "/wsp_model", "--select-ids", SEL, "--think", MODE, "--llm-dtype", DTYPE,
         "--gpus", GPUS, "--budget", str(BUDGET), "--top-layers", str(TOP_LAYERS), "--min-d", "0",
         "--deadline-ts", str(DEADLINE), "--round-sec-init", "600"])
BEST = json.load(open(S + "/best_spec.json")) if os.path.exists(S + "/best_spec.json") else []
pack()

In [ ]:
# Bước 3: bảng các bản sửa đã thử (trên 10 mẫu chọn)
rows = [json.loads(l) for l in open(S + "/wsp_log.jsonl", encoding="utf-8")]
base = rows[0]
print(f"{'vòng':>4s} {'bản sửa (tầng, cột, alpha)':32s} {'mean':>7s} {'so vòng 0':>9s} {'gen_tokens':>10s} {'phần tử đổi':>11s}  flags")
for r in rows:
    name = str([(o['layer'], o['top_k_cols'], o['alpha']) for o in r['spec']]) if r["spec"] else "gốc (vòng 0)"
    print(f"{r['round']:4d} {name[:32]:32s} {r['mean_score']:7.4f} {r['mean_score'] - base['mean_score']:+9.4f} "
          f"{str(r['gen_tokens_mean']):>10s} {r['n_changed']:11d}  {r['flags']}")
tried = [r for r in rows if r["spec"]]
if tried:
    top = max(tried, key=lambda r: r["mean_score"])
    json.dump(top["spec"], open(W + "/best_tried_spec.json", "w"), indent=1)
    print(f"\nĐã chạy {len(tried)} bản sửa. Bản điểm cao nhất: vòng {top['round']}, mean {top['mean_score']} (vòng 0: {base['mean_score']}).")
print("best_spec (qua luật giữ):", BEST if BEST else "[] = chưa có bản nào tăng >= 0,01 mà không tăng gen_tokens quá 5%")
print(f"Đã chạy {(time.time() - NB_T0) / 3600:.2f} giờ")

In [ ]:
# Bước 4 (tuỳ chọn): kiểm bản được giữ trên 10 mẫu dev. Bỏ qua nếu không có bản được giữ hoặc đã muộn.
el = (time.time() - NB_T0) / 3600
if BEST and el < DEV_CHECK_BEFORE_H:
    json.dump(BEST, open(W + "/best_spec.json", "w"), indent=1)
    run(["optimize/wsp/run_round.py", "--round", "0", "--spec", "none", "--think", MODE, "--ids", DEV, "--work", f"{W}/dev_{MODE}"] + K)
    run(["optimize/wsp/run_round.py", "--round", "1", "--spec", W + "/best_spec.json", "--dirs", W + "/dirs.pt",
         "--think", MODE, "--ids", DEV, "--work", f"{W}/dev_{MODE}"] + K)
    for r in [json.loads(l) for l in open(f"{W}/dev_{MODE}/wsp_log.jsonl", encoding="utf-8")]:
        print("dev 10 mẫu |", "WSP" if r["spec"] else "gốc", "| mean", r["mean_score"], "| gen_tokens", r["gen_tokens_mean"], "|", r["flags"])
else:
    print(f"Bỏ qua kiểm dev: best_spec={'có' if BEST else 'không'}, đã chạy {el:.2f} giờ (mốc {DEV_CHECK_BEFORE_H}).")
pack()